# RAG, properly: embeddings and a vector database

[`why_rag.ipynb`](why_rag.ipynb) ended on a cheat. We selected the relevant document
with this line:

```python
relevant = [d for d in KNOWLEDGE_BASE if "on-call" in d.lower()]
```

That is not retrieval, it is a substring match against a list we had already read. This
notebook replaces it with the real thing: an embedding model that understands *meaning*,
and a vector database that can search millions of documents in milliseconds.

Requirements: `ollama serve`, a chat model, an embedding model
(`ollama pull qwen3-embedding:0.6b`), and `chromadb`.

In [1]:
import time

import chromadb
import numpy as np
import ollama

CHAT_MODEL = "qwen3:30b-a3b-instruct-2507-q4_K_M"
EMBED_MODEL = "qwen3-embedding:0.6b"

KNOWLEDGE_BASE = [
    "Access to the Helios staging cluster is granted only through the VPN; direct SSH "
    "from outside the office network is refused at the firewall.",
    "Runbooks for the nightly ETL jobs live in the ops-handbook repository, under "
    "docs/runbooks/.",
    "On-call for the data platform is owned by Mirembe Okonkwo, mirembe@helios.internal.",
    "Laptop disk encryption keys are escrowed automatically; IT can recover a locked "
    "device within one business day.",
    "Expense claims above 400 EUR require a second approver before finance releases them.",
    "The staff shuttle departs the north entrance every 20 minutes, 07:40 to 09:40.",
    "Meeting room Basalt holds the only conference camera that drives two screens.",
    "Internal API tokens expire after 90 days and are rotated from the developer portal.",
    "The archive storage tier is read-only; restoring a file from it takes up to four hours.",
    "New starters collect their badge from reception on the first morning, not from IT.",
]
print(f"{len(KNOWLEDGE_BASE)} documents")

10 documents


## 1. Why keyword matching is not enough

Real users do not phrase questions using the words in your documents. They describe
their problem.

In [2]:
def keyword_search(query, documents):
    """Return documents sharing a substantial word with the query."""
    words = {w.strip("?,.").lower() for w in query.split() if len(w) > 4}
    return [d for d in documents if any(w in d.lower() for w in words)]


REAL_QUESTIONS = [
    "my laptop is locked, can I get my files back?",
    "how do I get a new API credential?",
    "how long does it take to get an old file back?",
]

for q in REAL_QUESTIONS:
    hits = keyword_search(q, KNOWLEDGE_BASE)
    print(
        f"{q}\n  -> {len(hits)} hits: {hits[0][:60] + '...' if hits else 'NOTHING FOUND'}\n"
    )

my laptop is locked, can I get my files back?
  -> 1 hits: Laptop disk encryption keys are escrowed automatically; IT c...

how do I get a new API credential?
  -> 0 hits: NOTHING FOUND

how long does it take to get an old file back?
  -> 0 hits: NOTHING FOUND



The first question got lucky — *"locked"* happens to appear in the document. The other
two returned nothing at all, even though the answers are sitting right there: *"how do I
get a new API credential?"* shares no word with *"internal API tokens are rotated from the
developer portal"*, and *"an old file"* shares none with *"the archive storage tier"*.

This is the **vocabulary mismatch** problem. Users describe their situation; documents
use institutional language; the two rarely overlap. To bridge it we have to compare
meanings, not strings.

## 2. Embeddings: meaning as coordinates

An **embedding model** maps a piece of text to a fixed-length vector of numbers — here
1024 of them. It is trained so that texts with similar meaning land near each other,
whatever words they use.

Once text is a vector, "similar meaning" becomes ordinary geometry. We normalise every
vector to length 1, which makes the **dot product** equal the cosine of the angle
between them: 1.0 is identical direction, 0.0 unrelated.

In [3]:
def embed(texts):
    """Embed a list of strings; return an (N, 1024) array of unit vectors."""
    vectors = np.array(ollama.embed(model=EMBED_MODEL, input=texts).embeddings)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


doc_vectors = embed(KNOWLEDGE_BASE)
print(f"{len(KNOWLEDGE_BASE)} documents -> array{doc_vectors.shape}")
print(f"every row has length {np.linalg.norm(doc_vectors[0]):.1f}")
print(f"\nfirst 8 numbers of document 2:\n{doc_vectors[2][:8].round(3)}")

10 documents -> array(10, 1024)
every row has length 1.0

first 8 numbers of document 2:
[-0.004 -0.055 -0.007 -0.113 -0.035 -0.075 -0.026 -0.056]


In [4]:
question = "my laptop is locked, can I get my files back?"
similarity = embed([question])[0] @ doc_vectors.T  # one dot product per document

for rank, i in enumerate(similarity.argsort()[::-1][:3], start=1):
    print(f"{rank}. {similarity[i]:.3f}  {KNOWLEDGE_BASE[i][:72]}...")

1. 0.674  Laptop disk encryption keys are escrowed automatically; IT can recover a...
2. 0.499  The archive storage tier is read-only; restoring a file from it takes up...
3. 0.376  Internal API tokens expire after 90 days and are rotated from the develo...


The correct document comes first, with no shared keywords at all. That single matrix
multiply — one query vector against every document vector — is the whole of semantic
search.

Which raises the obvious question: if it is one matmul over a NumPy array, why does
anyone need a database?

## 3. Why a database and not a list

For ten documents, you do not. `doc_vectors` above is a perfectly good index, and
[`scripts/mini_rag.py`](scripts/mini_rag.py) ships exactly that as its default backend.

A vector database earns its place when:

| | a NumPy array | a vector DB |
| --- | --- | --- |
| Survives restart | no — re-embed everything each run | yes, on disk |
| Adding one document | rebuild the array | one `upsert` |
| Search cost | scans every row | approximate index (HNSW), sublinear |
| Filtering by metadata | write it yourself | `where={"team": "data"}` |
| Ten million rows | will not fit in RAM | designed for it |

The first two matter immediately. Embedding is the slow, expensive step — doing it once
and keeping the result is the point. We will use **Chroma**, which runs in-process with
no server to manage.

In [5]:
# Scale up so the numbers mean something: the real facts plus filler.
def padded(n):
    filler = [
        f"Policy note {i}: equipment in lab {i % 9} is booked through the facilities "
        f"portal, and bookings lapse if unconfirmed {2 + i % 5} days ahead."
        for i in range(n - len(KNOWLEDGE_BASE))
    ]
    return KNOWLEDGE_BASE + filler


CORPUS = padded(800)
client = chromadb.PersistentClient(path="./chroma_helios")
client.delete_collection("helios") if "helios" in [
    c for c in client.list_collections()
] else None

collection = client.get_or_create_collection(
    "helios", metadata={"hnsw:space": "cosine"}
)
print(f"collection '{collection.name}' holds {collection.count()} documents")

collection 'helios' holds 0 documents


In [6]:
# Embed once, in batches, and store. This is the slow step -- and the one we never
# have to repeat, because the collection is on disk.
if collection.count() < len(CORPUS):
    started = time.perf_counter()
    for start in range(0, len(CORPUS), 100):
        batch = CORPUS[start : start + 100]
        collection.upsert(
            ids=[f"doc-{start + j}" for j in range(len(batch))],
            documents=batch,
            embeddings=embed(batch).tolist(),
            metadatas=[
                {"source": "handbook" if start + j < len(KNOWLEDGE_BASE) else "filler"}
                for j in range(len(batch))
            ],
        )
    print(
        f"indexed {collection.count()} documents in {time.perf_counter() - started:.1f}s"
    )
else:
    print(f"already indexed: {collection.count()} documents")

indexed 800 documents in 9.9s


In [7]:
def retrieve(query, k=3):
    """Return the k most semantically similar documents."""
    result = collection.query(query_embeddings=embed([query]).tolist(), n_results=k)
    return result["documents"][0], result["distances"][0]


started = time.perf_counter()
docs, distances = retrieve("who is responsible for the data platform rota?")
print(
    f"searched {collection.count()} documents in {(time.perf_counter() - started) * 1000:.0f}ms\n"
)
for d, dist in zip(docs, distances, strict=True):
    print(f"  {1 - dist:.3f}  {d[:70]}...")

searched 800 documents in 102ms

  0.678  On-call for the data platform is owned by Mirembe Okonkwo, mirembe@hel...
  0.487  Internal API tokens expire after 90 days and are rotated from the deve...
  0.455  Policy note 504: equipment in lab 0 is booked through the facilities p...


## 4. The full RAG loop

Everything is now in place. Retrieval-augmented generation is three steps, and the whole
system is the function below.

In [8]:
def rag(question, k=3, show=True):
    documents, _ = retrieve(question, k=k)  # 1. retrieve
    context = "\n".join(f"- {d}" for d in documents)  # 2. augment
    system = (
        "You are the internal helpdesk assistant at Helios. Answer briefly and directly. "
        "Use only the context below; if the answer is not there, say you do not know.\n\n"
        f"Context:\n{context}"
    )
    response = ollama.chat(  # 3. generate
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
    )
    if show:
        print(f"Q: {question}")
        print(f"A: {response['message']['content'].strip()}")
        print(
            f"   [{response['prompt_eval_count']} prompt tokens, {k} of "
            f"{collection.count()} documents sent]\n"
        )
    return response


for q in REAL_QUESTIONS + ["who is responsible for the data platform rota?"]:
    rag(q)

Q: my laptop is locked, can I get my files back?
A: Yes, your laptop disk encryption keys are escrowed automatically, so IT can recover your device within one business day. Once recovered, you should be able to access your files.
   [135 prompt tokens, 3 of 800 documents sent]



Q: how do I get a new API credential?
A: You can rotate your API token from the developer portal. Tokens expire after 90 days.
   [139 prompt tokens, 3 of 800 documents sent]



Q: how long does it take to get an old file back?
A: Up to four hours, as the archive storage tier is read-only and restoration takes up to four hours.
   [122 prompt tokens, 3 of 800 documents sent]



Q: who is responsible for the data platform rota?
A: Mirembe Okonkwo is responsible for the data platform rota.
   [137 prompt tokens, 3 of 800 documents sent]



Every one of those was answered from the handbook, and none of them would have been
found by keyword matching.

The cost comparison against stuffing the whole corpus, from part 1:

| approach | prompt tokens | scales to 1M documents? |
| --- | --- | --- |
| Whole corpus in the prompt | ~24,700 | no — exceeds any context window |
| Retrieve top 3, then generate | ~200 | yes — the prompt size never changes |

That is the payoff. **The prompt no longer grows with the corpus.** A million documents
cost the same per question as ten, because only the three that matter are ever sent.

## 5. It is not magic

Embeddings fail too, and it is worth seeing one fail rather than trusting the good
results above.

In [9]:
docs, distances = retrieve("who do I page at 3am?", k=3)
for d, dist in zip(docs, distances, strict=True):
    print(f"  {1 - dist:.3f}  {d[:70]}...")

  0.467  Policy note 480: equipment in lab 3 is booked through the facilities p...
  0.465  Policy note 705: equipment in lab 3 is booked through the facilities p...
  0.464  Policy note 300: equipment in lab 3 is booked through the facilities p...


The on-call document does not come back at all. Instead we get three near-identical
filler documents at ~0.46, and the winning margin between them is 0.002 — the model is
not discriminating, it is returning noise.

*"Page at 3am"* and *"owns on-call"* are the same idea to you. To a 0.6B embedding model
they are not: it has not reliably learned that operational jargon, so nothing in the
corpus stands out and the ranking collapses to near-ties. A flat, low top score like this
is the signal worth watching for — it usually means *nothing relevant was found*, which
is a very different situation from a confident top hit.

This is the honest state of semantic search, and it is exactly what the rest of the
pipeline in [`index.ipynb`](index.ipynb) exists to repair:

* **Hybrid retrieval** — run keyword (BM25) and vector search together and fuse the
  rankings, so a rare exact term like a name or an error code is not lost to fuzzy
  matching.
* **Reranking** — retrieve 50 candidates cheaply, then rescore them with a
  cross-encoder that reads query and document *together*. Far more accurate, far too
  slow to run over the whole corpus.
* **Chunking** — our documents are one sentence each. Real documents are long and must
  be split, and how you split them bounds everything downstream.
* **A better embedding model** — 0.6B is small. Larger models handle jargon better.

The general lesson: **retrieval quality is the ceiling on RAG quality.** A perfect
generator cannot answer from a document it was never given.

## 6. What the database gives you

Two things a NumPy array does not, both visible now that the data is on disk.

In [10]:
# Persistence: a fresh client, no re-embedding, in a new process this would be identical.
reopened = chromadb.PersistentClient(path="./chroma_helios").get_collection("helios")
print(f"reopened without re-embedding: {reopened.count()} documents\n")

# Metadata filters: search only part of the corpus.
filtered = reopened.query(
    query_embeddings=embed(["who is on call?"]).tolist(),
    n_results=2,
    where={"source": "handbook"},
)
print("restricted to source='handbook':")
for d in filtered["documents"][0]:
    print(f"  {d[:70]}...")

reopened without re-embedding: 800 documents

restricted to source='handbook':
  On-call for the data platform is owned by Mirembe Okonkwo, mirembe@hel...
  Meeting room Basalt holds the only conference camera that drives two s...


Incremental updates work the same way — `upsert` a changed document by its id and only
that one is re-embedded. Compare that with re-running the whole pipeline because one
policy page changed.

### Where to go next

| notebook | what it covers |
| --- | --- |
| [`why_rag.ipynb`](why_rag.ipynb) | why stuffing the prompt stops working |
| **this one** | embeddings, a vector DB, the basic RAG loop |
| [`index.ipynb`](index.ipynb) | chunking, hybrid search, reranking, evaluation |
| [`scripts/mini_rag.py`](scripts/mini_rag.py) | the whole pipeline as one runnable file |

The `./chroma_helios` directory on disk is the index — delete it to start over.